<a href="https://colab.research.google.com/github/JakeOh/202511_BD53/blob/main/lab_ml/ml21_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM(Large Language Model, 거대 언어 모델)

*   시퀀스-시퀀스 작업(Sequence-to-Sequence)
    *   시퀀스 데이터(sequential data)르 입력으로 받아서 시퀀스 데이터를 출력하는 작업.
    *   자연어 처리(NLP, Natural Language Processing) 분야에서 요약, 번역 등의 작업.
    *   두 개의 (순환) 신경망 연결한 인코더-디코더(encoder-decoder) 구조가 널리 사용됨.
*   어텐션 메커니즘(Attention mechanism)
    *   인코더-디코더 구조에서 사용된 순환 신경망(RNN)의 성능을 향상시키기 위해서 고안.
    *   기존에는 인코더의 마지막 타입 스텝에서 출력한 은닉 상태만을 사용해서 디코더가 새로운 텍스트를 생성.
    *   어텐션 메커니즘은 모든 타임 스텝에서 인코더가 출력한 은닉 상태를 디코더가 참조할 수 있도록 고안.
    *   디코더 새로운 토큰을 생성할 때 인코더가 처리한 토큰들 중에서 어떤 토큰에 더 주의(attention)를 기울 지를 결정. 입력 토큰들마다 디코더가 중요도를 다르게 부여.
*   트랜스포머 모델(Transformer model)
    *   어텐션 메커니즘을 기반으로 해서 인코더-디코더 구조에서 순환층을 제거.
    *   인코더에서 한 번에 하나의 토큰을 처리하지 않고 입력 텍스트 전체를 한 번에 처리.
    *   핵심 구성 요서
        *   멀티 헤드 어텐션(multi-head attention)
        *   층 정규화(layer normalization)
        *   잔차 연결(residual connection)
        *   피드 포워드 네트워크(feed-forward network)

<img src="https://pbs.twimg.com/media/GJg3QtMXwAAvUT4?format=jpg&name=large"
    alt="LLM 발전 과정" />

*   오른쪽 회색 가지
    *   Transformer 모델을 사용하지 않은 알고리즘.
    *   워드 임베딩 벡터를 만드는 알고리즘.
*   Encoder 기반 모델
    *   텍스트 (긍정/부정) 분류
    *   개체명 인식 - 텍스트에서 사람 이름, 지역 이름, 회사 이름 등의 고유 명사를 식별.
    *   BERT, RoBERTa, ALBERT, ...
*   Encoder-Decoder 기반 모델
    *   문서 요약, 번역, 질문-답변
    *   T5, BART, ...
*   Decoder 기반 모델
    *   텍스트 생성 - 챗봇, 질문-답변, 요약, 번역, ...
    *   디코더
        *   이전까지 생성한 텍스트를 입력받아서 다음 토큰을 예측하는 방식.
        *   인코더로부터 입력이 없으면 디코더는 아무것도 생성할 수 없음.
        *   이전에 생성한 텍스트인 것처럼 어떤 텍스트를 입력해주면 인코더 도움 없이 다음 토큰을 예측할 수 있음.
        *   프롬프트(prompt): 이전에 생성된 텍스트인 것처럼 전달하는 초기 텍스트.
    *   GPT-4, GPT-5, LLaMA, Claude, ...
    *   현재 가장 활발히 연구되는 LLM 분야.

# BART 모델을 사용한 문서 요약

Hugging Face - 오픈 소스 LLM 모델(Transformer 기반)들 제공. Transformer 모델들을 사용할 수 있는 패키지 제공.

In [1]:
import transformers  # Hugging Face에서 제공하는 Transformer 모델들을 사용할 수 있는 패키지

In [2]:
transformers.__version__

'5.0.0'

*   2026/03/23 현재 transformers 패키지의 가장 최신 안정(stable)적인 버전은 5.3.0
    *   Google Colab에 설치된 버전 5.0.0
*   transformers 패키지 5.0.0 버전부터는 pipeline 함수에서 "summarization" task(작업)이 삭제됨.
*   BART 모델 소개를 위해서 이전 버전의 transformer 패키지를 Colab에 설치.
    *   4.x 버전에서 가장 마지막 안정(stable) 버전은 4.57.6

In [ ]:
# 코드 셀에서 Linux 명령어 실행하기
# pip: Python 패키지 관리자 명령어. 패키지 설치/삭제/업데이트.
# pip install [옵션] 패키지이름[==특정버전]
!pip install transformers==4.57.6

In [2]:
import transformers

In [3]:
transformers.__version__

'4.57.6'

In [ ]:
# Hugging Face에서 오픈 소스로 공개된 모델을 다운로드해서 로컬에서 실행할 수 있도록 함.
# pipeline() 함수 호출 - Pipeline 객체 생성
# pipeline() 함수의 model 파라미터를 설정하지 않으면 "sshleifer/distilbart-cnn-12-6" 모델을 기본값으로 다운로드
# pipeline() 함수의 device 파라미터: device=-1(기본값, CPU 사용), device=0(GPU 계산)
pipe = transformers.pipeline(task="summarization")

In [5]:
print(type(pipe))

<class 'transformers.pipelines.text2text_generation.SummarizationPipeline'>


In [6]:
sample_text = '''
The 2026 Iran war began on 28 February 2026, when the United States and Israel launched surprise airstrikes on multiple sites and cities across Iran, killing Supreme Leader Ali Khamenei and several other Iranian officials. Iran responded with missile and drone strikes against Israel, US bases, and US-allied countries in the Middle East.

After the Middle Eastern crisis began in 2023, Iran and Israel exchanged missile strikes in 2024, and Israel and the US launched airstrikes against Iran in the Twelve-Day War in June 2025. In January 2026, Iranian security forces killed thousands of protesters during the largest protests since the Iranian Revolution. US president Donald Trump responded by threatening military action against Iran and launching the largest U.S. military buildup in the region since the US-led 2003 invasion of Iraq. In mid-February, the U.S. and Iran began a new round of indirect nuclear negotiations.[67]

The surprise US-Israeli attack, launched during the nuclear negotiations, killed Khamenei, other Iranian officials, and civilians.[68][69] Subsequent attacks damaged military bases, government facilities, schools, hospitals, and cultural heritage sites.[70] In retaliation, Iran launched hundreds of drones and ballistic missiles at targets in Israel and at US military bases in Bahrain, Jordan, Kuwait, Qatar, Saudi Arabia, Turkey, and the United Arab Emirates.[71][72] A drone struck Britain's Akrotiri military base on Cyprus. Other strikes hit civilian infrastructure including in Azerbaijan,[73] Kurdistan, and Oman.[74] Iran denied striking Turkey, Oman, and Azerbaijan, saying that they were "Israeli false flag” attacks.[75][76] Two ballistic missiles were reportedly launched at Diego Garcia military base on the Chagos Islands,[77][78] which Iran also denied.[79] The conflict between Hezbollah and Israel escalated into the 2026 Lebanon war.[80][81]

Trump administration officials have offered various and conflicting explanations for starting the war,[82] such as to ward off an imminent Iranian threat, to pre-empt Iranian retaliation against US assets after an expected Israeli attack on Iran,[83][84][85][86] to destroy Iran's missile and military capabilities, to prevent Iran from obtaining a nuclear weapon,[87] to secure Iran's natural resources,[88] and to achieve regime change by bringing the Iranian opposition to power.[89][90][91][92] Iran, as well as officials from the Pentagon, rejected claims that Iran had been preparing an attack.[93] The International Atomic Energy Agency (IAEA) said that it did not have the access it needed to ensure that the Iranian nuclear program was exclusively peaceful, but that there was no evidence of a structured nuclear weapons program at the time of the strikes.[94] UN secretary-general Antonio Guterres and several uninvolved countries condemned the US–Israeli strikes; the United Nations Security Council later passed a resolution condemning Iran's retaliatory strikes on the Gulf states.[95] Critics of the war, including legal and international relations experts, have described the attacks as illegal under US law, an act of imperialism and a violation of Iran's sovereignty[96][97] under international law.[98][99] On 19 March, the cost of the war to the US was estimated at US$18 billion,[100] and the Pentagon requested an additional $200 billion for the war.[101][102]

The war's economic impact, described as the world's largest supply disruption since the 1970s energy crisis,[103] includes surges in oil and gas prices, widespread disruptions in aviation and tourism, and heightened volatility in financial markets. Contributors include Iran's closure of the Strait of Hormuz, seen by some as a violation of the law of the sea, and Israel and Iran's attacks on energy facilities, both disrupting global oil and gas shipments.[104][105] On 23 March 2026, President Trump postponed US strikes against Iranian power plants for five days amid alleged negotiations.[106]
'''

In [7]:
# Pipeline 객체를 함수처럼 호출 - sample_text를 요약
result = pipe(sample_text)

In [8]:
print(result)

[{'summary_text': ' The 2026 Iran war began on 28 February 2026, when the United States and Israel launched surprise airstrikes on multiple sites and cities across Iran, killing Supreme Leader Ali Khamenei and several other Iranian officials . Iran responded with missile and drone strikes against Israel, US bases, and US-allied countries in the Middle East . The cost of the war to the US was estimated at US$18 billion, and the Pentagon requested an additional $200 billion for the war .'}]


pipe() 함수 호출의 리턴 값은 dict를 아이템으로 갖는 list.

list가 가지고 있는 dict 개수는 1개.

dict는 (key-value) 아이템을 1개만 갖고 있음.

In [9]:
len(result)

1

In [10]:
result[0]  #> list의 첫번째 원소 -> dict(key-value 아이템 1개)

{'summary_text': ' The 2026 Iran war began on 28 February 2026, when the United States and Israel launched surprise airstrikes on multiple sites and cities across Iran, killing Supreme Leader Ali Khamenei and several other Iranian officials . Iran responded with missile and drone strikes against Israel, US bases, and US-allied countries in the Middle East . The cost of the war to the US was estimated at US$18 billion, and the Pentagon requested an additional $200 billion for the war .'}

In [13]:
result[0]['summary_text']  # 요약 내용

' The 2026 Iran war began on 28 February 2026, when the United States and Israel launched surprise airstrikes on multiple sites and cities across Iran, killing Supreme Leader Ali Khamenei and several other Iranian officials . Iran responded with missile and drone strikes against Israel, US bases, and US-allied countries in the Middle East . The cost of the war to the US was estimated at US$18 billion, and the Pentagon requested an additional $200 billion for the war .'

위에서 사용한 LLM 모델 "sshleifer/distilbart-cnn-12-6"은 영어만 훈련된 모델. 한글 텍스트를 요약하려고 하면 에러가 발생. 한글 요약 작업을 하기 위해서는 한글로 훈련된 모델을 다운로드해서 실행해야 됨.

## KoBART 모델을 사용한 한글 문서 요약

In [ ]:
pipe_kor = transformers.pipeline(task='summarization',              # task=작업내용
                                 model='EbanLee/kobart-summary-v3', # model=다운로드받을 모델명
                                 device=-1                          # device=GPU 사용 여부
                                 )

In [15]:
sample_text_kor = '''
2026년 이란 전쟁(영어: 2026 Iran war)은 이란 이슬람 공화국의 정권 교체 및 핵 저지를 목표로 한 서방의 선제 타격과, 이에 맞선 이란 및 대리 세력의 보복이 맞물려 확대된 국제전이다. 미국-이란 전쟁 또는 중동 전쟁으로 부른다. 미국과 이스라엘은 이란의 핵 개발을 저지하고 지역 내 영향력을 차단하기 위해 최고 지도자 하메네이를 비롯한 수뇌부 암살과 주요 시설 공습을 단행했으며, 이에 이란과 이른바 '저항의 축'이라 불리는 대리 세력이 호르무즈 해협을 봉쇄하고 중동 내 미군 기지를 공격하며 강력하게 대응하면서 전쟁의 양상이 전방위적으로 확산되었다.

2026년 2월 28일, 이스라엘과 미국은 이란 전역의 여러 지점과 도시에 기습 공습을 감행하여, 최고 지도자 알리 하메네이와 다수의 이란 관료들을 살해했다. 이란은 이스라엘, 미군 기지 및 지역 내 미국의 우방국들을 향해 미사일과 드론 공격으로 대응했다.

2023년 중동 위기가 시작된 이후, 이란과 이스라엘은 2024년에 미사일 공격을 주고받았으며, 2025년 6월 12일 전쟁 당시에도 충돌이 있었다. 이 과정에서 미국이 이란 핵 시설을 공습하기도 했다. 2026년 1월, 이란 보안군은 이슬람 혁명 이후 최대 규모의 시위 도중 수천 명의 시위대를 살해했다. 도널드 트럼프 미국 대통령은 이러한 살해 행위에 대해 이란 정권에 군사적 조치를 취하겠다고 위협했다. 이란과 미국은 2월에 간접적인 핵 협상을 진행했으나,[63] 동시에 미국은 2003년 이라크 침공 이후 중동에서 최대 규모의 군사력 증강에 착수했다.

재개된 미국-이란 핵 협상 기간 중 미국과 이스라엘이 감행한 합동 공습으로 이란의 최고 지도자 알리 하메네이가 암살되었으며, 그의 관저가 파괴되었다. 다른 이란 관료들도 살해되었다.[64][65] 이후 이란은 수백 대의 드론과 탄도 미사일을 이스라엘의 표적과 바레인, 요르단, 쿠웨이트, 카타르, 사우디아라비아, 튀르키예, 그리고 아랍에미리트에 위치한 미군 기지를 향해 발사했다.[66][67][68] 이란 내에서는 민간인 사상자가 발생했으며 학교, 병원, 문화유산 유적지 등이 피해를 입었다.[69] 이란의 한 학교 건물은 미국의 미사일에 맞아 100명 이상의 초등학생과 50~100명의 다른 사람들이 사망했다.[70] 이란은 아제르바이잔, 쿠르디스탄, 오만의 민간인 목표물을 타격했다.[71] 키프로스섬에 있는 영국의 아크로티리 군사 기지는 드론 공격을 받았다.[72] 2024년 이후 이스라엘의 레바논 내 헤즈볼라 공습으로 국한되었던 헤즈볼라와 이스라엘 사이의 갈등은, 헤즈볼라가 이스라엘 군사 기지에 미사일과 드론 발사를 재개하고 이스라엘이 레바논에 대한 공습을 대폭 확대하며 베이루트 폭격을 재개함에 따라 2026년 레바논 전쟁으로 격화되었다.[73][74]

미국은 자국 작전의 목적을 명확히 밝히지 않았다.[75] 도널드 트럼프 대통령과 다른 미국 관리들은 이번 공격에 대해 임박한 이란의 위협을 막기 위함이라는 것과, 불가피한 이스라엘의 이란 공격 이후 미국 자산에 대한 이란의 보복을 선제적으로 차단하기 위함이라는 것을 포함해 여러 가지 이유를 제시했다.[76][77][78][79] 또한 이란의 미사일 및 군사 능력을 파괴하고, 이란이 핵무기를 보유하는 것을 영구적으로 방지하며,[80] 천연자원을 확보하고,[81] 궁극적으로는 정권 교체를 달성하여 이란 반대파를 집권시키기 위함이라는 이유도 포함되었다.[82][83][84][85] 이란뿐만 아니라 펜타곤의 관리들도 이란이 공격을 준비하고 있었다는 주장을 부인했다.[86][87] 국제 원자력 기구(IAEA)는 이란의 핵 프로그램이 전적으로 평화적인지 확인하는 데 필요한 접근 권한을 갖지 못했으나, 공습 당시 조직적인 핵무기 프로그램의 증거는 없었다고 밝혔다.[88] 안토니우 구테흐스 유엔 사무총장과 여러 비교전국들은 미국과 이스라엘의 공습을 규탄했으며, 유엔 안전 보장 이사회는 이후 이란의 걸프 국가들에 대한 보복 공격을 규탄하는 결의안을 통과시켰다.[89] 법학 전문가를 포함한 초기 공습 비판론자들은 이번 공격을 국제법에 따른 이란의 주권 침해로 규정했다.[90][91][92]

이 갈등은 즉각적인 유가 및 천연가스 가격의 급등, 항공 및 관광업의 광범위한 혼란, 금융 시장의 변동성 심화를 초래했다.[93][94] 이란은 호르무즈 해협을 강제로 폐쇄하고 에너지 시설을 공격하여 전 세계 석유 및 가스 선적에 차질을 빚게 했다.[95]
'''

In [16]:
result = pipe_kor(sample_text_kor)

In [17]:
print(result)

[{'summary_text': '이란 전쟁은 이란 이슬람 공화국의 정권 교체 및 핵 저지를 목표로 한 서방의 선제 타격과, 이에 맞선 이란 및 대리 세력의 보복이 맞물려 확대된 국제전이다. 미국과 이스라엘은 이란의 핵 개발을 저지하고 지역 내 영향력을 차단하기 위해 최고 지도자 하메네이를 비롯한 수뇌부 암살과 주요 시설 공습을 단행했으며, 이에 이란과 대리 세력이 중동 내 미군 기지를 공격하며 강력하게 대응하면서 전쟁의 양상이 전방위적으로 확산되었다.'}]


In [18]:
result[0]['summary_text']

'이란 전쟁은 이란 이슬람 공화국의 정권 교체 및 핵 저지를 목표로 한 서방의 선제 타격과, 이에 맞선 이란 및 대리 세력의 보복이 맞물려 확대된 국제전이다. 미국과 이스라엘은 이란의 핵 개발을 저지하고 지역 내 영향력을 차단하기 위해 최고 지도자 하메네이를 비롯한 수뇌부 암살과 주요 시설 공습을 단행했으며, 이에 이란과 대리 세력이 중동 내 미군 기지를 공격하며 강력하게 대응하면서 전쟁의 양상이 전방위적으로 확산되었다.'

# BART 모델 동작 원리

*   encoder-decoder 모델
*   텍스트 입력 --> encode() --> LLM 모델 --> decode() --> 텍스트 출력
*   pipeline() 함수는 LLM 모델과 텍스트를 토큰화하는 tokenizer 객체를 함께 다운로드함.
*   tokenizer 객체가 텍스트 인코딩과 디코딩을 수행하는 메서드를 가지고 있음.

In [19]:
tokenizer = pipe_kor.tokenizer
print(type(tokenizer))

<class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>


In [20]:
# tokenize() 메서드: 텍스트 -> 토큰들의 리스트
tokens = tokenizer.tokenize('혼자 공부하는 머신러닝 딥러닝')
print(tokens)

['▁혼자', '▁공부', '하는', '▁머', '신', '러', '닝', '▁', '딥', '러', '닝']


In [21]:
# convert_tokens_to_ids(): 토큰들의 리스트를 인코딩된 정수(인덱스)들의 리스트로 변환
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[16814, 16962, 14049, 14771, 11467, 10277, 9747, 1700, 10021, 10277, 9747]


In [22]:
# convert_ids_to_token(): 토큰 아이디(인덱스)들의 리스트를 해당하는 토큰들의 리스트로 변환
result = tokenizer.convert_ids_to_tokens(ids)
print(result)

['▁혼자', '▁공부', '하는', '▁머', '신', '러', '닝', '▁', '딥', '러', '닝']


In [23]:
# encode(): tokenize + convert_tokens_to_ids
encoded = tokenizer.encode('KoBART 모델을 사용한 문서 요약')
print(encoded)
# encode() 메서드는 텍스트 시작(0)과 끝(1)을 나타내는 아이디가 포함.

[0, 14572, 310, 265, 264, 281, 283, 24224, 21032, 26052, 26200, 1]


In [24]:
tokenizer.convert_ids_to_tokens(encoded)

['<s>', '▁K', 'o', 'B', 'A', 'R', 'T', '▁모델을', '▁사용한', '▁문서', '▁요약', '</s>']

In [25]:
# decode(): 토큰 아이디들의 리스트를 텍스트로 변환
decoded = tokenizer.decode(encoded)
print(decoded)
#> 디코딩된 텍스트에는 시작(<s>)과 끝(</s>)을 나타내는 특수기호가 포함.

<s> KoBART 모델을 사용한 문서 요약</s>


In [32]:
encoded[0]  # 인코딩된 아이디 리스트의 첫번째 원소는 항상 0 - 텍스트 시작을 의미

0

In [34]:
encoded[-1]  # 인코딩된 아이디 리스트의 마지막 원소는 항상 1 - 텍스트 끝을 의미

1

In [36]:
tokenizer.decode(encoded[1:-1])

'KoBART 모델을 사용한 문서 요약'